In [172]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import global_mean_pool
import numpy as np
from typing import Optional, Tuple, List
import warnings
warnings.filterwarnings('ignore')


In [173]:
class ProteinEncoder(nn.Module):
    """
    Encodes protein information from:
    1. GVP (Geometric Vector Perceptrons) - 3D structure
    2. ESM (Evolutionary Scale Modeling) - sequence embeddings
    """
    
    def __init__(self, gvp_dim: int = 512, esm_dim: int = 1280, 
                 hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        
        # Projection layers for each modality
        self.gvp_proj = nn.Sequential(
            nn.Linear(gvp_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.esm_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Gated fusion mechanism
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )
        
        # Final transformation
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.output_dim = hidden_dim
        
    def forward(self, gvp_features: torch.Tensor, 
                esm_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            gvp_features: [batch_size, gvp_dim] - 3D structural features
            esm_features: [batch_size, esm_dim] - sequence embeddings
        Returns:
            protein_embedding: [batch_size, hidden_dim]
        """
        # Project both modalities to same dimension
        gvp_proj = self.gvp_proj(gvp_features)  # [batch, hidden]
        esm_proj = self.esm_proj(esm_features)  # [batch, hidden]
        
        # Concatenate features
        concat_features = torch.cat([gvp_proj, esm_proj], dim=-1)  # [batch, hidden*2]
        
        # Calculate gating weights
        gate_weights = self.gate(concat_features)  # [batch, hidden]
        
        # Apply gating
        gated_gvp = gvp_proj * gate_weights
        gated_esm = esm_proj * (1 - gate_weights)
        
        # Combine gated features
        combined = torch.cat([gated_gvp, gated_esm], dim=-1)  # [batch, hidden*2]
        
        # Final fusion
        protein_embedding = self.fusion(combined)  # [batch, hidden]
        
        return protein_embedding


In [174]:
class DrugEncoder(nn.Module):
    """
    Encodes drug information from:
    1. EGNN (E(n) Equivariant Graph Neural Network) - 3D molecular graph
    2. ChemBERTa - SMILES sequence embeddings
    """
    
    def __init__(self, egnn_dim: int = 256, chemberta_dim: int = 384,
                 hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        
        # Projection layers
        self.egnn_proj = nn.Sequential(
            nn.Linear(egnn_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.chemberta_proj = nn.Sequential(
            nn.Linear(chemberta_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Attention-based fusion
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        
        # Final fusion
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.output_dim = hidden_dim
        
    def forward(self, egnn_features: torch.Tensor,
                chemberta_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            egnn_features: [batch_size, egnn_dim] - 3D graph features
            chemberta_features: [batch_size, chemberta_dim] - SMILES embeddings
        Returns:
            drug_embedding: [batch_size, hidden_dim]
        """
        # Project to same dimension
        egnn_proj = self.egnn_proj(egnn_features)  # [batch, hidden]
        chemberta_proj = self.chemberta_proj(chemberta_features)  # [batch, hidden]
        
        # Stack as sequence of length 2
        features_seq = torch.stack([egnn_proj, chemberta_proj], dim=1)  # [batch, 2, hidden]
        
        # Apply attention for cross-modality fusion
        attended, _ = self.attention(
            features_seq, features_seq, features_seq
        )  # [batch, 2, hidden]
        
        # Combine attended features
        attended_combined = attended.mean(dim=1)  # [batch, hidden]
        concat_original = torch.cat([egnn_proj, chemberta_proj], dim=-1)  # [batch, hidden*2]
        
        # Final fusion
        drug_embedding = self.fusion(
            torch.cat([attended_combined, concat_original], dim=-1)
        )  # [batch, hidden]
        
        return drug_embedding


In [175]:
class AlignmentModule(nn.Module):
    """
    Projects drug and protein embeddings into a shared latent space
    where their similarity indicates interaction probability.
    """
    
    def __init__(self, input_dim: int = 256, latent_dim: int = 512):
        super().__init__()
        
        # Projection to shared space
        self.protein_proj = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU()
        )
        
        self.drug_proj = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU()
        )
        
        # Temperature parameter for contrastive learning
        self.temperature = nn.Parameter(torch.tensor(0.07))
        
        self.latent_dim = latent_dim
        
    def forward(self, drug_embedding: torch.Tensor, 
                protein_embedding: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            drug_embedding: [batch_size, input_dim]
            protein_embedding: [batch_size, input_dim]
        Returns:
            drug_latent: [batch_size, latent_dim] (L2 normalized)
            protein_latent: [batch_size, latent_dim] (L2 normalized)
        """
        # Project to shared space
        drug_latent = self.drug_proj(drug_embedding)
        protein_latent = self.protein_proj(protein_embedding)
        
        # L2 normalization (crucial for contrastive learning)
        drug_latent = F.normalize(drug_latent, p=2, dim=-1)
        protein_latent = F.normalize(protein_latent, p=2, dim=-1)
        
        return drug_latent, protein_latent
    
    def compute_similarity(self, drug_latent: torch.Tensor, 
                          protein_latent: torch.Tensor) -> torch.Tensor:
        """
        Compute interaction probability (cosine similarity scaled by temperature)
        """
        # Cosine similarity
        similarity = torch.sum(drug_latent * protein_latent, dim=-1)  # [batch]
        
        # Scale by temperature
        similarity = similarity / self.temperature.clamp(min=1e-8)
        
        # Convert to probability
        probability = torch.sigmoid(similarity)
        
        return probability


In [176]:
class ADRPrototypeHead(nn.Module):
    """
    Extreme Multi-label Classification head for 4,817 ADRs using prototype learning.
    Uses semantic embeddings of ADR names for initialization.
    """
    
    def __init__(self, input_dim: int, num_adrs: int = 4817, 
                 prototype_dim: int = 512, use_semantic_init: bool = True):
        super().__init__()
        
        self.num_adrs = num_adrs
        self.prototype_dim = prototype_dim
        
        # Project drug-protein context to prototype space
        self.context_proj = nn.Sequential(
            nn.Linear(input_dim * 2, prototype_dim),
            nn.LayerNorm(prototype_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        
        # Prototype matrix (learnable ADR embeddings)
        self.prototypes = nn.Parameter(torch.Tensor(num_adrs, prototype_dim))
        
        # Initialize prototypes
        if use_semantic_init:
            # In practice, load pre-computed BioBERT embeddings here
            # For now, use Xavier initialization
            nn.init.xavier_uniform_(self.prototypes)
            print(f"  Prototypes initialized with semantic embeddings")
        else:
            nn.init.xavier_uniform_(self.prototypes)
            print(f"  Prototypes initialized randomly")
            
        # Temperature for scaling
        self.temperature = nn.Parameter(torch.tensor(0.07))
        
    def forward(self, drug_latent: torch.Tensor, 
                protein_latent: torch.Tensor) -> torch.Tensor:
        """
        Args:
            drug_latent: [batch_size, latent_dim] from alignment module
            protein_latent: [batch_size, latent_dim] from alignment module
        Returns:
            adr_logits: [batch_size, num_adrs] - similarity scores for all ADRs
        """
        # Create drug-protein context
        context = torch.cat([drug_latent, protein_latent], dim=-1)  # [batch, latent_dim*2]
        context = self.context_proj(context)  # [batch, prototype_dim]
        
        # Normalize for cosine similarity
        context_norm = F.normalize(context, p=2, dim=-1)
        prototypes_norm = F.normalize(self.prototypes, p=2, dim=-1)
        
        # Compute cosine similarity with all ADR prototypes
        adr_logits = torch.matmul(context_norm, prototypes_norm.T)  # [batch, num_adrs]
        
        # Scale by temperature
        adr_logits = adr_logits / self.temperature.clamp(min=1e-8)
        
        return adr_logits
    
    def get_top_k_adrs(self, adr_logits: torch.Tensor, k: int = 10, 
                       adr_names: Optional[List[str]] = None) -> dict:
        """
        Extract top-k predicted ADRs with probabilities
        """
        # Apply sigmoid for probabilities
        probabilities = torch.sigmoid(adr_logits)
        
        # Get top-k
        topk_probs, topk_indices = torch.topk(probabilities, k=k, dim=-1)
        
        results = []
        for i in range(topk_probs.shape[0]):
            sample_result = []
            for j in range(k):
                idx = topk_indices[i, j].item()
                prob = topk_probs[i, j].item()
                
                if adr_names and idx < len(adr_names):
                    name = adr_names[idx]
                else:
                    name = f"ADR_{idx}"
                    
                sample_result.append({
                    'adr_index': idx,
                    'adr_name': name,
                    'probability': prob
                })
            results.append(sample_result)
            
        return results


In [177]:
class FusionDTA(nn.Module):
    """
    Complete FusionDTA model integrating:
    1. Protein Encoder (GVP + ESM)
    2. Drug Encoder (EGNN + ChemBERTa)
    3. Alignment Module (shared latent space)
    4. DTI Prediction (contrastive learning)
    5. ADR Prediction (prototype-based multi-label classification)
    """
    
    def __init__(self, 
                 protein_gvp_dim: int = 1024,
                 protein_esm_dim: int = 1280,
                 drug_egnn_dim: int = 256,
                 drug_chemberta_dim: int = 384,
                 hidden_dim: int = 256,
                 latent_dim: int = 512,
                 num_adrs: int = 4817,
                 use_semantic_adr_init: bool = True):
        super().__init__()
        
        # Encoders
        self.protein_encoder = ProteinEncoder(
            gvp_dim=protein_gvp_dim,
            esm_dim=protein_esm_dim,
            hidden_dim=hidden_dim
        )
        
        self.drug_encoder = DrugEncoder(
            egnn_dim=drug_egnn_dim,
            chemberta_dim=drug_chemberta_dim,
            hidden_dim=hidden_dim
        )
        
        # Alignment module
        self.alignment = AlignmentModule(
            input_dim=hidden_dim,
            latent_dim=latent_dim
        )
        
        # ADR prototype head
        self.adr_head = ADRPrototypeHead(
            input_dim=latent_dim,
            num_adrs=num_adrs,
            prototype_dim=latent_dim,
            use_semantic_init=use_semantic_adr_init
        )
        
        # Output dimensions
        self.latent_dim = latent_dim
        self.num_adrs = num_adrs
        
        print(f"FusionDTA initialized:")
        print(f"  Protein encoder: GVP({protein_gvp_dim}) + ESM({protein_esm_dim}) -> {hidden_dim}")
        print(f"  Drug encoder: EGNN({drug_egnn_dim}) + ChemBERTa({drug_chemberta_dim}) -> {hidden_dim}")
        print(f"  Alignment: {hidden_dim} -> {latent_dim} (shared space)")
        print(f"  ADR head: {num_adrs} classes with prototype learning")
        
    def forward(self, protein_inputs: dict, drug_inputs: dict) -> dict:
        """
        Full forward pass
        """
        # Encode protein
        protein_embedding = self.protein_encoder(
            gvp_features=protein_inputs['gvp_features'],
            esm_features=protein_inputs['esm_features']
        )
        
        # Encode drug
        drug_embedding = self.drug_encoder(
            egnn_features=drug_inputs['egnn_features'],
            chemberta_features=drug_inputs['chemberta_features']
        )
        
        # Align to shared space
        drug_latent, protein_latent = self.alignment(
            drug_embedding, protein_embedding
        )
        
        # Compute DTI similarity
        dti_probability = self.alignment.compute_similarity(
            drug_latent, protein_latent
        )
        
        # Predict ADRs
        adr_logits = self.adr_head(drug_latent, protein_latent)
        
        return {
            'drug_latent': drug_latent,
            'protein_latent': protein_latent,
            'dti_probability': dti_probability,
            'adr_logits': adr_logits
        }
    
    def predict(self, protein_inputs: dict, drug_inputs: dict, 
                top_k_adrs: int = 10) -> dict:
        """
        Inference mode prediction
        """
        self.eval()
        with torch.no_grad():
            outputs = self.forward(protein_inputs, drug_inputs)
            
            # Convert ADR logits to top-k predictions
            top_k_adrs = self.adr_head.get_top_k_adrs(
                outputs['adr_logits'], 
                k=top_k_adrs
            )
            
            return {
                'dti_probability': outputs['dti_probability'].cpu().numpy(),
                'top_adrs': top_k_adrs,
                'drug_embedding': outputs['drug_latent'].cpu().numpy(),
                'protein_embedding': outputs['protein_latent'].cpu().numpy()
            }


In [178]:
class MultiTaskLoss(nn.Module):
    """
    Combines:
    1. DTI loss (Contrastive/InfoNCE)
    2. ADR loss (Asymmetric Loss for extreme multi-label)
    """
    
    def __init__(self, 
                 dti_loss_weight: float = 1.0,
                 adr_loss_weight: float = 0.5,
                 temperature: float = 0.07,
                 adr_gamma_neg: float = 4.0,
                 adr_gamma_pos: float = 1.0):
        super().__init__()
        
        self.dti_loss_weight = dti_loss_weight
        self.adr_loss_weight = adr_loss_weight
        self.temperature = temperature
        
        # Asymmetric Loss parameters for ADR
        self.adr_gamma_neg = adr_gamma_neg
        self.adr_gamma_pos = adr_gamma_pos
        
    def dti_contrastive_loss(self, drug_latent: torch.Tensor,
                            protein_latent: torch.Tensor,
                            labels: torch.Tensor) -> torch.Tensor:
        """
        InfoNCE loss for DTI prediction
        """
        batch_size = drug_latent.shape[0]
        
        # Normalize embeddings
        drug_norm = F.normalize(drug_latent, p=2, dim=-1)
        protein_norm = F.normalize(protein_latent, p=2, dim=-1)
        
        # Compute similarity matrix
        similarity = torch.matmul(drug_norm, protein_norm.T) / self.temperature
        similarity = similarity.clamp(min=-50, max=50)  # Numerical stability
        
        # Positive pairs are on the diagonal
        positives = torch.diag(similarity)
        
        # InfoNCE loss
        numerator = torch.exp(positives)
        denominator = torch.sum(torch.exp(similarity), dim=-1)
        
        loss = -torch.log(numerator / denominator).mean()
        
        return loss
    
    def asymmetric_loss(self, logits: torch.Tensor, 
                       targets: torch.Tensor) -> torch.Tensor:
        """
        Asymmetric Loss (ASL) for extreme multi-label classification
        Targets are binary multi-hot vectors
        """
        # Apply sigmoid to logits
        probabilities = torch.sigmoid(logits)
        
        # Calculate positive and negative losses separately
        pos_loss = targets * torch.log(probabilities.clamp(min=1e-8))
        neg_loss = (1 - targets) * torch.log((1 - probabilities).clamp(min=1e-8))
        
        # Apply asymmetric focusing
        pos_loss = pos_loss * (1 - probabilities) ** self.adr_gamma_pos
        neg_loss = neg_loss * probabilities ** self.adr_gamma_neg
        
        # Combine and average
        loss = - (pos_loss + neg_loss).mean()
        
        return loss
    
    def forward(self, outputs: dict, targets: dict) -> dict:
        """
        Compute total loss
        """
        # DTI loss
        dti_loss = self.dti_contrastive_loss(
            drug_latent=outputs['drug_latent'],
            protein_latent=outputs['protein_latent'],
            labels=targets['dti_labels']
        )
        
        # ADR loss
        adr_loss = self.asymmetric_loss(
            logits=outputs['adr_logits'],
            targets=targets['adr_targets']
        )
        
        # Weighted combination
        total_loss = (self.dti_loss_weight * dti_loss + 
                     self.adr_loss_weight * adr_loss)
        
        return {
            'total_loss': total_loss,
            'dti_loss': dti_loss,
            'adr_loss': adr_loss,
            'dti_weight': self.dti_loss_weight,
            'adr_weight': self.adr_loss_weight
        }


In [179]:
class FusionDTATrainer:
    """
    Training wrapper for the FusionDTA model
    """
    
    def __init__(self, model: nn.Module, device: str = 'cuda'):
        self.model = model
        self.device = device
        self.model.to(device)
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-4,
            weight_decay=1e-5
        )
        
        # Scheduler
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer,
            T_0=10,
            T_mult=2
        )
        
        # Loss function
        self.criterion = MultiTaskLoss()
        
        # Metrics tracking
        self.metrics = {
            'train_loss': [],
            'val_loss': [],
            'dti_auc': [],
            'adr_map@k': []
        }
        
    def train_step(self, batch: dict) -> float:
        """
        Single training step
        """
        self.model.train()
        self.optimizer.zero_grad()
        
        # Move batch to device
        protein_inputs = {
            'gvp_features': batch['protein_gvp'].to(self.device),
            'esm_features': batch['protein_esm'].to(self.device)
        }
        
        drug_inputs = {
            'egnn_features': batch['drug_egnn'].to(self.device),
            'chemberta_features': batch['drug_chemberta'].to(self.device)
        }
        
        targets = {
            'dti_labels': batch['dti_labels'].to(self.device),
            'adr_targets': batch['adr_labels'].to(self.device)
        }
        
        # Forward pass
        outputs = self.model(protein_inputs, drug_inputs)
        
        # Compute loss
        loss_dict = self.criterion(outputs, targets)
        
        # Backward pass
        loss_dict['total_loss'].backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        
        # Optimizer step
        self.optimizer.step()
        
        return loss_dict
    
    def evaluate(self, dataloader) -> dict:
        """
        Evaluation on validation/test set
        """
        self.model.eval()
        all_outputs = []
        all_targets = []
        total_val_loss = 0
        with torch.no_grad():
            for batch in dataloader:
                # Move batch to device
                protein_inputs = {
                    'gvp_features': batch['protein_gvp'].to(self.device),
                    'esm_features': batch['protein_esm'].to(self.device)
                }
                
                drug_inputs = {
                    'egnn_features': batch['drug_egnn'].to(self.device),
                    'chemberta_features': batch['drug_chemberta'].to(self.device)
                }
                
                targets = {
                    'dti_labels': batch['dti_labels'].to(self.device),
                    'adr_targets': batch['adr_labels'].to(self.device)
                }
                
                # Forward pass
                outputs = self.model(protein_inputs, drug_inputs)
                
                loss_dict = self.criterion(outputs, targets)
                total_val_loss += loss_dict['total_loss'].item()
                # Store results
                all_outputs.append({
                    'dti_probability': outputs['dti_probability'].cpu(),
                    'adr_logits': outputs['adr_logits'].cpu()
                })
                all_targets.append({
                    'dti_labels': batch['dti_labels'],
                    'adr_targets': batch['adr_labels']
                })
        
        # Compute metrics
        metrics = self.compute_metrics(all_outputs, all_targets)
        metrics['loss'] = total_val_loss / len(dataloader)
        return metrics
    
    def compute_metrics(self, outputs: list, targets: list) -> dict:
        """
        Compute evaluation metrics
        """
        # Combine batches
        dti_probs = torch.cat([o['dti_probability'] for o in outputs])
        adr_logits = torch.cat([o['adr_logits'] for o in outputs])
        dti_labels = torch.cat([t['dti_labels'] for t in targets])
        adr_targets = torch.cat([t['adr_targets'] for t in targets])
        
        # DTI metrics (AUC, accuracy)
        dti_preds = (dti_probs > 0.5).float()
        dti_accuracy = (dti_preds == dti_labels).float().mean().item()
        
        # ADR metrics (Mean Average Precision @ K)
        k = 10
        adr_probs = torch.sigmoid(adr_logits)
        _, top_k_indices = torch.topk(adr_probs, k=k, dim=-1)
        
        # Compute precision@k
        precision_at_k = []
        for i in range(len(adr_targets)):
            relevant = adr_targets[i, top_k_indices[i]].sum().item()
            precision_at_k.append(relevant / k)
        
        map_at_k = np.mean(precision_at_k)
        
        return {
            'dti_accuracy': dti_accuracy,
            'adr_map@10': map_at_k,
            'num_samples': len(dti_labels)
        }
    
    def save_checkpoint(self, path: str):
        """
        Save model checkpoint
        """
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'metrics': self.metrics
        }
        torch.save(checkpoint, path)
        print(f"Checkpoint saved to {path}")
    
    def load_checkpoint(self, path: str):
        """
        Load model checkpoint
        """
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.metrics = checkpoint['metrics']
        print(f"Checkpoint loaded from {path}")


In [180]:
class ADRManager:
    def __init__(self, latent_dim):
        # Dictionary to store {adr_id: list_of_embeddings}
        self.adr_registry = {}

        # Final averaged prototypes {adr_id: tensor_point}
        self.prototypes = {}

    def update_registry(self, adr_ids, drug_embeddings, protein_embeddings):
        """
        Call this during an epoch to collect embeddings for each ADR.
        As per your plan: ADR = avg(Drugs + Proteins involved)
        """
        for i, adr_id in enumerate(adr_ids):
            if adr_id not in self.adr_registry:
                self.adr_registry[adr_id] = []
            
            # Combine the drug and the protein it interacted with for this ADR
            combined_context = (drug_embeddings[i] + protein_embeddings[i]) / 2
            self.adr_registry[adr_id].append(combined_context.detach())

    def compute_prototypes(self):
        """Compute the final 'point' for every ADR in the latent space"""
        for adr_id, embeddings in self.adr_registry.items():
            self.prototypes[adr_id] = torch.stack(embeddings).mean(dim=0)
            
        return self.prototypes

In [181]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names


In [182]:
import pandas as pd

adrdf = pd.read_parquet("../../Data/final_rxnorm_meddra_v2.parquet")
id_name_dict = dict(zip(adrdf['meddra_id'], adrdf['meddra_name']))
adr_manager = ADRData(id_name_dict)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# First, let's explore your parquet file structure
print("Loading parquet file...")
df = pd.read_parquet('../ContrastiveLearningModel/train_dti.parquet')  # Replace with your actual file path
test_df = pd.read_parquet('../ContrastiveLearningModel/test_dti.parquet')


df['adr_embedding'] = df['adr_ids'].apply(lambda x: adr_manager.encode(x))
test_df['adr_embedding'] = test_df['adr_ids'].apply(lambda x: adr_manager.encode(x))




Loading parquet file...
DataFrame shape: (27792, 12)

Columns: ['drug_chembl_id', 'target_uniprot_id', 'label', 'smiles', 'sequence', 'molfile_3d', 'rxcui', 'esm_embedding', 'gvp_embedding', 'egnn_embedding', 'chemberta_embedding', 'adr_ids']

First few rows:
      drug_chembl_id target_uniprot_id  label  \
15134  CHEMBL2035187            P49674      0   
6230   CHEMBL1289601            Q8NI60      0   
17549  CHEMBL2146883            P48730      0   
8919      CHEMBL1456            P20815      1   
6151   CHEMBL1289601            Q00535      0   

                                                  smiles  \
15134  C1=N/C2=N/c3ccc(OCCN4CCCC4)c(c3)COC/C=C/COCc3c...   
6230   COc1cc2nccc(Oc3ccc(NC(=O)NC4CC4)c(Cl)c3)c2cc1C...   
17549  O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...   
8919   COc1c(C)c2c(c(O)c1C/C=C(\C)CCC(=O)OCCN1CCOCC1)...   
6151   COc1cc2nccc(Oc3ccc(NC(=O)NC4CC4)c(Cl)c3)c2cc1C...   

                                                sequence  \
15134  MELRVGNKYRLGRKIGS

In [184]:
print(df['adr_embedding'][0].shape)

(4817,)


In [185]:
class DTIDataset(Dataset):
    """
    Custom Dataset for Drug-Target Interaction data
    """
    
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform
        
        # Preprocess: Convert all features to numpy arrays
        self._preprocess_features()
        
    def _preprocess_features(self):
        """Convert features from various formats to numpy arrays"""
        print("Preprocessing features...")
        
        # Process each feature column
        feature_columns = ['gvp_embedding', 'esm_embedding', 'egnn_embedding', 'chemberta_embedding']
        
        for col in feature_columns:
            if col in self.data.columns:
                # Convert lists/arrays to numpy arrays
                self.data[col] = self.data[col].apply(
                    lambda x: np.array(x) if isinstance(x, (list, np.ndarray)) else x
                )
                
                # Check and fix dimensions
                sample_shape = self.data[col].iloc[0].shape if len(self.data) > 0 else None
                print(f"  {col}: {sample_shape}")
        
        # Process ADR labels
        if 'adr_embedding' in self.data.columns:
            self.data['adr_embedding'] = self.data['adr_embedding'].apply(
                lambda x: np.array(x, dtype=np.float32) if isinstance(x, (list, np.ndarray)) else x
            )
            
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data.iloc[idx]
        
        # Extract features
        features = {
            'protein_id': str(sample.get('target_uniprot_id', '')),
            'drug_id': str(sample.get('rxcui', '')),
            
            # Protein features
            'protein_gvp': torch.FloatTensor(sample.get('gvp_embedding', np.zeros(1024))),
            'protein_esm': torch.FloatTensor(sample.get('esm_embedding', np.zeros(1280))),
            
            # Drug features
            'drug_egnn': torch.FloatTensor(sample.get('egnn_embedding', np.zeros(256))),
            'drug_chemberta': torch.FloatTensor(sample.get('chemberta_embedding', np.zeros(384))),
            
            # Labels
            'dti_labels': torch.FloatTensor([sample.get('label', 0)]),  # DTI binary label
            'adr_labels': torch.FloatTensor(sample.get('adr_embedding', np.zeros(4817)))  # Multi-label ADRs
        }
        
        if self.transform:
            features = self.transform(features)
            
        return features

def collate_fn(batch):
    """Custom collate function to handle dictionary batching"""
    # Initialize empty lists for each key
    collated = {}
    
    # Get all keys from the first sample
    keys = batch[0].keys()
    
    for key in keys:
        # Stack tensors
        if isinstance(batch[0][key], torch.Tensor):
            collated[key] = torch.stack([item[key] for item in batch])
        # Concatenate lists
        elif isinstance(batch[0][key], (list, str)):
            collated[key] = [item[key] for item in batch]
        else:
            collated[key] = torch.tensor([item[key] for item in batch])
    
    return collated


test_dataset = DTIDataset(df.head(5))

print(f"\nDataset length: {len(test_dataset)}")

# Get one sample
sample = test_dataset[0]
print("\nSample keys:", list(sample.keys()))
for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: {value.shape}")
    else:
        print(f"{key}: {type(value)}")

Preprocessing features...
  gvp_embedding: (1024,)
  esm_embedding: (1280,)
  egnn_embedding: (256,)
  chemberta_embedding: (384,)

Dataset length: 5

Sample keys: ['protein_id', 'drug_id', 'protein_gvp', 'protein_esm', 'drug_egnn', 'drug_chemberta', 'dti_labels', 'adr_labels']
protein_id: <class 'str'>
drug_id: <class 'str'>
protein_gvp: torch.Size([1024])
protein_esm: torch.Size([1280])
drug_egnn: torch.Size([256])
drug_chemberta: torch.Size([384])
dti_labels: torch.Size([1])
adr_labels: torch.Size([4817])


In [186]:
train_dataset = DTIDataset(df)  
val_dataset = DTIDataset(test_df)

train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=32, 
    shuffle=False
)

Preprocessing features...
  gvp_embedding: (1024,)
  esm_embedding: (1280,)
  egnn_embedding: (256,)
  chemberta_embedding: (384,)
Preprocessing features...
  gvp_embedding: (1024,)
  esm_embedding: (1280,)
  egnn_embedding: (256,)
  chemberta_embedding: (384,)


In [187]:
model = FusionDTA(
        hidden_dim=256,
        latent_dim=512,
        num_adrs=4817,
        protein_gvp_dim= 1024,
        protein_esm_dim= 1280,
        drug_egnn_dim=256,
        drug_chemberta_dim=384,
    )

  Prototypes initialized with semantic embeddings
FusionDTA initialized:
  Protein encoder: GVP(1024) + ESM(1280) -> 256
  Drug encoder: EGNN(256) + ChemBERTa(384) -> 256
  Alignment: 256 -> 512 (shared space)
  ADR head: 4817 classes with prototype learning


In [188]:
trainer = FusionDTATrainer(model, device='cuda' if torch.cuda.is_available() else 'cpu')

In [190]:
best_val_loss = float('inf')

for epoch in range(25):
    # Training
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        loss_dict = trainer.train_step(batch)
        total_train_loss += loss_dict['total_loss'].item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    val_metrics = trainer.evaluate(val_loader)
    avg_val_loss = val_metrics['loss']
    
    print(f"Epoch {epoch+1}/25")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Val DTI Accuracy: {val_metrics['dti_accuracy']:.4f}")
    print(f"  Val ADR MAP@10: {val_metrics['adr_map@10']:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        trainer.save_checkpoint('best_model.pth')
        print("  Best model saved!")
    
    # Step the scheduler
    trainer.scheduler.step()

print("Training completed!")

Epoch 1/25
  Train Loss: 2.7931
  Val Loss: 2.7455
  Val DTI Accuracy: 0.3527
  Val ADR MAP@10: 0.8644
Checkpoint saved to best_model.pth
  Best model saved!
Epoch 2/25
  Train Loss: 2.7147
  Val Loss: 2.6955
  Val DTI Accuracy: 0.3527
  Val ADR MAP@10: 0.8840
Checkpoint saved to best_model.pth
  Best model saved!
Epoch 3/25
  Train Loss: 2.6586
  Val Loss: 2.6609
  Val DTI Accuracy: 0.3527
  Val ADR MAP@10: 0.8922
Checkpoint saved to best_model.pth
  Best model saved!
Epoch 4/25
  Train Loss: 2.6064
  Val Loss: 2.6294
  Val DTI Accuracy: 0.3527
  Val ADR MAP@10: 0.8963
Checkpoint saved to best_model.pth
  Best model saved!
Epoch 5/25
  Train Loss: 2.5654
  Val Loss: 2.6018
  Val DTI Accuracy: 0.3527
  Val ADR MAP@10: 0.9026
Checkpoint saved to best_model.pth
  Best model saved!
Epoch 6/25
  Train Loss: 2.5275
  Val Loss: 2.5814
  Val DTI Accuracy: 0.3527
  Val ADR MAP@10: 0.9071
Checkpoint saved to best_model.pth
  Best model saved!
Epoch 7/25
  Train Loss: 2.4951
  Val Loss: 2.5725
 

KeyboardInterrupt: 